# <b> PART 1 </b>
### GROUP 2 - 24CTT3

### <b> Helper Function </b>

In [21]:
import copy
def augmented_matrix(A: list, b: list):
    """INPUT: Matrix A, b in Ax = b
       Output: A_augmented = (A|B)"""
    return [row + [val] for row, val in zip(A, b)]

def change_row(A: list, idx1 : int, idx2: int):
    """Truyền idx1, idx2 theo số bắt đầu từ 1 cho giống với làm bài bình thường, đỡ nhầm"""
    if idx1 > len(A) or idx2 > len(A):
        return -1
    temp = A[idx1]
    A[idx1] = A[idx2]
    A[idx2] = temp

def multiply_row_with_c(A: list, idx1 : int, c: int):
    """Nhân một dòng với số c"""
    if idx1> len(A):
        return -1

    for i in range(len(A[idx1])):
        A[idx1][i] *= c

def plus_another_row(A: list, idx1: int, idx2: int, c: int):
    """Input: Ma trận A, idx1 là hàng được cộng, c là số nhân
    R_idx1 <- R_idx1 + c*R_idx2"""
    A_copy = copy.deepcopy(A)
    multiply_row_with_c(A_copy, idx2, c)

    for i in range(len(A[idx1])):
        A[idx1][i] += A_copy[idx2][i]

def isREF(A: list):
    """Check xem đã phải ma trận bậc thang chưa"""
    prev_pivot = -1
    for i in range(len(A)):
        curr_pivot = -1
        for j in range(len(A[0])):
            if abs(A[i][j]) > 1e-10:
                curr_pivot = j
                break

        if curr_pivot == -1:
            prev_pivot = float('inf')
        else:
            if curr_pivot <= prev_pivot:
                return False
            prev_pivot = curr_pivot
    return True

### <b> Hàm giải hệ tam giác trên </b>
<b> Note: </b> Hàm trả về list nghiệm nếu có đúng 1 nghiệm, trả về "Vô nghiệm", hoặc trả về ["Vô số nghiệm" , nghiệm có chuẩn nhỏ nhất]

In [22]:
def back_substitution(U: list, c: list):
    EPS = 1e-10

    m = len(U)
    if m == 0: return []
    n = len(U[0]) # Kích thước m x n

    # x[i] lưu biểu thức của x_i
    x = [[0.0] * (n + 1) for _ in range(n)]

    # Giả định ban đầu mọi biến đều là "ẩn tự do" (ví dụ: x2 = 1*x2)
    for i in range(n):
        x[i][i + 1] = 1.0

    # Chạy ngược từ dòng cuối lên trên
    for i in range(m - 1, -1, -1):
        # Tìm phần tử (pivot) của dòng i
        p = -1
        for j in range(n):
            if abs(U[i][j]) > EPS:
                p = j
                break

        # Xử lý trường hợp dòng toàn số 0 (không có pivot)
        if p == -1:
            if abs(c[i]) > EPS:
                return "Hệ vô nghiệm"
            continue

        # (Fix 1) Tránh chia cho số rất nhỏ
        if abs(U[i][p]) < EPS:
            continue

        # Rút biến cơ sở x_p theo các biến đằng sau nó
        x[p] = [0.0] * (n + 1)
        x[p][0] = c[i] / U[i][p]

        # Thế biểu thức của các x_j vào
        for j in range(p + 1, n):
            if abs(U[i][j]) > EPS:
                factor = -U[i][j] / U[i][p]
                # Cộng dồn hệ số: x_p = x_p + factor * x_j
                for k in range(n + 1):
                    x[p][k] += factor * x[j][k]

    # Chuyển đổi mảng hệ số thành chuỗi công thức tổng quát
    ket_qua = []
    for i in range(n):
        tmp = x[i]

        # (Fix 2) Không so sánh float trực tiếp
        if abs(tmp[i + 1] - 1.0) < EPS and all(abs(tmp[k]) < EPS for k in range(n + 1) if k != i + 1):
            ket_qua.append(f"x_{i+1} (ẩn tự do)")
            continue

        terms = []
        if abs(tmp[0]) > EPS:
            terms.append(f"{tmp[0]:.2f}")

        for k in range(1, n + 1):
            if abs(tmp[k]) > EPS:
                sign = "+" if tmp[k] > 0 else "-"
                val = f"{abs(tmp[k]):.2f}" if abs(abs(tmp[k]) - 1.0) > EPS else ""
                terms.append(f"{sign} {val}x{k}")

        bieu_thuc = " ".join(terms).strip()
        if bieu_thuc.startswith("+ "):
            bieu_thuc = bieu_thuc[2:]
        elif not bieu_thuc:
            bieu_thuc = "0"

        ket_qua.append(f"x_{i+1} = {bieu_thuc}")

    return ket_qua

### <b> Hàm giải hệ phương trình tuyến tính </b>
Hàm giải hệ phương trình tuyến tính Ax = b, trả về: Ma trận tăng cường, nghiệm x, số lần biến đổi

In [23]:
def gaussian_eliminate(A: list, b: list):
    """Hàm chính: Giải hệ phương trình Ax = b"""

    EPS = 1e-10  # (Fix 3)

    augMatrix = augmented_matrix(A, b)

    ref_matrix = copy.deepcopy(augMatrix)
    n = len(ref_matrix)
    m_total = len(ref_matrix[0])

    row, col, swap_count = 0, 0, 0

    while row < n and col < m_total:
        maxVal = -1
        max_row = -1

        for i in range(row, n):
            if abs(ref_matrix[i][col]) > maxVal:
                maxVal = abs(ref_matrix[i][col])
                max_row = i

        if abs(ref_matrix[max_row][col]) < EPS:
            col += 1
            continue

        if max_row != row:
            change_row(ref_matrix, row, max_row)
            swap_count += 1

        # (Fix 1) Tránh chia cho pivot rất nhỏ
        if abs(ref_matrix[row][col]) < EPS:
            col += 1
            continue

        for i in range(row + 1, n):
            # (Fix 1) tránh chia cho số rất nhỏ
            if abs(ref_matrix[row][col]) < EPS:
                continue
            c_factor = -ref_matrix[i][col] / ref_matrix[row][col]
            plus_another_row(ref_matrix, i, row, c_factor)

        row += 1
        col += 1

    n, m = len(A), len(A[0])

    for i in range(n):
        is_zero_row_A = True
        for j in range(m):
            if abs(ref_matrix[i][j]) > EPS:
                is_zero_row_A = False
                break

        if is_zero_row_A and abs(ref_matrix[i][m]) > EPS:
            return ref_matrix, "Vô nghiệm", swap_count

    U = [row[:m] for row in ref_matrix]
    c_vec = [row[m] for row in ref_matrix]

    result = back_substitution(U, c_vec)

    return ref_matrix, result, swap_count

### <b> Hàm tính định thức của ma trận vuông </b>

In [24]:
def determinant(A):
    n = len(A)
    dummy_b = [0] * n

    M, _, s = gaussian_eliminate(A, dummy_b)

    det = (-1) ** s
    for i in range(n):
        det *= M[i][i]

    return det

A = [[2, 1], [1,1]]
print(determinant(A))

1.0


### <b> Rank and basis </b>

In [25]:
def rank_and_basis(A):
    nRow = len(A)
    nCol = len(A[0])

    #Khử Gauss lấy ma trận bậc thang
    B = [0.0] * nRow
    augMatrix, _, _ = gaussian_eliminate(A, B)

    #Xác định rank và vị trí cac cột pivot
    pivotCol = []
    currRow = 0
    rank = 0
    for i in range(nCol):
        if currRow < nRow and abs(augMatrix[currRow][i]) > 1e-10:
            pivotCol.append(i)
            rank += 1
            currRow += 1

    #Cơ sở ko gian dòng
    rowBasis = [augMatrix[i][:nCol] for i in range(rank)]

    #cơ sở ko gian cột
    colBasis = []
    for i in pivotCol:
        basisColVector = [A[j][i] for j in range(nRow)]
        colBasis.append(basisColVector)

    #cơ sở ko gian nghiệm
    nullBasis = []
    #chứa các chỉ số cột ko phải pivot
    freeVars = [i for i in range(nCol) if i not in pivotCol]
    for f in freeVars:
        c = [0.0] * nRow
        specialSol = [0.0] * nCol
        specialSol[f] = 1.0
        #Gán biến tu do =1
        for i in range(rank):
            c[i] = -augMatrix[i][f]
            #Chuyển hệ so bien tu do sang vế phải

        #Xuất ma trận vuông từ các cột chốt
        reducedMatrix = []
        for i in range(rank):
            reducedRow = [augMatrix[i][j] for j in pivotCol]
            reducedMatrix.append(reducedRow)
        #Giải hệ để tìm giá trị các bien chốt
        pivotVal = back_substitution(reducedMatrix, c[:rank])

        for i in range(len(pivotCol)):
            pCol = pivotCol[i]

            val = pivotVal[i]

            # Nếu là string dạng "x_1 = 1.00" thì parse lấy số
            if isinstance(val, str):
                val = float(val.split("=")[1])

            specialSol[pCol] = val

        nullBasis.append(specialSol)

    return rank, rowBasis, colBasis, nullBasis

### <b> Hàm tìm ma trận nghịch đảo </b>
Tìm ma trận nghịch đảo của ma trận A, trả về ma trận nghịch đảo hoặc các đoạn thông báo nếu không có ma trận nghịch đảo

In [26]:
def inverse(A : list):
    n = len(A)
    if any([len(A[i]) != n for i in range(n)]):
        return "Ma tran khong vuong, khong co nghich dao!"

    # Tao ma tran don vi I
    I = [[1.0 if i == j else 0.0 for j in range(n)] for i in range(n)]

    # Tao ma tran [A | I]
    M = [row_A + row_I for row_A, row_I in zip(A, I)]

    # Gauss-Jordan
    for i in range(n):
        # Chon phan tu chot (Partial Pivoting)
        pivot_row = i
        for j in range(i + 1, n):
            if abs(M[j][i]) > abs(M[pivot_row][i]):
                pivot_row = j

        M[i], M[pivot_row] = M[pivot_row], M[i]

        # Kiem tra ma tran suy bien
        if abs(M[i][i]) < 1e-10:
            return "Ma tran suy bien, khong co nghich dao!"

        # Chuan hoa dong chua pivot ve 1
        pivot_val = M[i][i]
        M[i] = [x / pivot_val for x in M[i]]

        # Khu cac phan tu khac tren cung cot ve 0
        for j in range(n):
            if i != j:
                factor = M[j][i]
                M[j] = [val_j - factor * val_i for val_i, val_j in zip(M[i], M[j])]

    # Trich xuat phan ma tran ben phai [I | A_inv]
    A_inv = [row[n:] for row in M]
    return A_inv

### Numpy Test
Sử dụng các hàm có sẵn của <b> Numpy </b> để kiểm tra tính đúng đắn của code

#### <b> Các hàm tính toán bằng Numpy </b>

In [56]:
import numpy as np

def back_substitution_np(U, c, tol=1e-10):
    U = np.array(U, float)
    c = np.array(c, float)

    rows, cols = U.shape
    x = np.zeros(cols)

    # Chỉ xử lý phần có pivot
    for i in range(min(rows, cols) - 1, -1, -1):

        if abs(U[i][i]) < tol:
            return None  # không xử lý suy biến / vô số nghiệm

        s = np.dot(U[i, i+1:], x[i+1:])
        x[i] = (c[i] - s) / U[i][i]

    return x

def solve_linear_system(A, b, tol=1e-10):
    A = np.array(A, float)
    b = np.array(b, float)

    n_rows, n_variables = A.shape

    rank_A = np.linalg.matrix_rank(A, tol)
    rank_aug = np.linalg.matrix_rank(np.c_[A, b], tol)

    if rank_A < rank_aug:
        return "Vô nghiệm"

    if rank_A == n_variables:
        if n_rows == n_variables:
            return np.linalg.solve(A, b)
        else:
            x, *_ = np.linalg.lstsq(A, b, rcond=None)
            return x

    # ===== CASE 3: VÔ SỐ NGHIỆM =====
    return "Vô số nghiệm"

def rank_np(A, tol=1e-10):
    return np.linalg.matrix_rank(A, tol)

def det_np(A):
    return float(np.linalg.det(A))

def inverse_np(A):
    A = np.array(A, dtype=float)
    if np.linalg.matrix_rank(A) < A.shape[0]:
        return "Không khả nghịch"
    return np.linalg.inv(A)

def null_space_np(A, tol=1e-10):
    A = np.array(A, dtype=float)
    u, s, vh = np.linalg.svd(A)

    rank = (s > tol).sum()
    null_space = vh[rank:].T   # (n x k)

    basis = []

    for i in range(null_space.shape[1]):
        v = null_space[:, i].copy()
        pivot = np.argmax(np.abs(v))

        if abs(v[pivot]) < tol:
            continue
        v = v / v[pivot]
        v[np.abs(v) < tol] = 0.0

        basis.append(v.tolist())

    return basis
def is_valid_solution(A, x, b):
    """Kiểm tra xem Ax có bằng b không (chấp nhận sai số nhỏ)"""
    if x is None or isinstance(x, str):
        return False
    A_np = np.array(A)
    x_np = np.array(x)
    b_np = np.array(b)
    # Kiểm tra Ax - b xấp xỉ 0
    return np.allclose(A_np @ x_np, b_np, atol=1e-8)


In [57]:
def verify_LS(LSTestCase: list):
  epsilon = 1e-10

  for i in range(len(LSTestCase)):
        A, b = LSTestCase[i]

        X_np = solve_linear_system(A, b)
        X_f = gaussian_eliminate(A, b)[1]

        # CASE 1: VÔ NGHIỆM
        if isinstance(X_np, str) and X_np == "Vô nghiệm":
            if not (isinstance(X_f, str) and X_f == "Vô nghiệm"):
                print(f"LS: Sai ở Test {i} (đáng ra vô nghiệm)")
            print(f"LS: Đúng ở Test {i}")
            continue

        if isinstance(X_f, str) and X_f == "Vô nghiệm":
            print(f"LS: Sai ở Test {i} (bị kết luận vô nghiệm sai)")
            continue

        # CASE 2: VÔ SỐ NGHIỆM
        if isinstance(X_np, str) and X_np == "Vô số nghiệm":
            if not (isinstance(X_f, list) and X_f[2] == "x_3 (ẩn tự do)"):
                print(f"LS: Sai ở Test {i} (không nhận diện vô số nghiệm)")
                print(X_f)
                continue
        n = len(X_f)
        x = [0.0] * n

        for idx, expr in enumerate(X_f):
            # Nếu là biến tự do gán = 0
            if "ẩn tự do" in expr:
                x[idx] = 0.0
            else:
                rhs = expr.split("=")[1].strip()

                val = 0.0
                terms = rhs.replace('-', '+-').split('+')

                for term in terms:
                    term = term.strip()
                    if not term:
                        continue

                    # Nếu là hằng số
                    if 'x' not in term:
                        val += float(term)
                    else:
                        # dạng: ax_k hoặc x_k
                        if 'x' in term:
                            if term.startswith('-'):
                                sign = -1
                                term = term[1:]
                            else:
                                sign = 1

                            if term.startswith('x'):
                                coef = 1.0
                                var_idx = int(term[1:]) - 1
                            else:
                                parts = term.split('x')
                                coef = float(parts[0])
                                var_idx = int(parts[1]) - 1

                            # vì x_free = 0 neen không cần cộng gì
                            val += sign * coef * 0.0

                x[idx] = val

        if not is_valid_solution(A, x, b):
            print(f"LS: Sai ở Test {i} (nghiệm không thỏa Ax=b)")
        else:
            print(f"LS: Đúng ở Test {i}")

        continue

        #CASE 3: NGHIỆM DUY NHẤT
        # X_np là numpy array
        if isinstance(X_f, list) and all(isinstance(s, str) for s in X_f):
            try:
                x = []
                for expr in X_f:
                    val = float(expr.split('=')[1])
                    x.append(val)

                if not is_valid_solution(A, x, b):
                    print(f"LS: Sai ở Test {i} (nghiệm sai)")
            except:
                print(f"LS: Sai ở Test {i} (parse lỗi)")
        else:
            print(f"LS: Sai ở Test {i} (format output sai)")
            print(X_np, X_f)

        print(f"LS: Đúng ở Test {i}")

In [58]:
def verify_inverse(InverseTestCase: list):
  epsilon = 1e9-10
  for i in range(len(InverseTestCase)):
        A = InverseTestCase[i]

        # Kết quả từ hàm do bạn code
        inv_f = inverse(A)

        # Kết quả chuẩn từ Numpy
        A_np = np.array(A, dtype=float)
        try:
            # np.linalg.inv sẽ văng lỗi LinAlgError nếu ma trận không vuông hoặc suy biến
            inv_np = np.linalg.inv(A_np)
            is_invertible = True
        except np.linalg.LinAlgError:
            is_invertible = False

        # Đối chiếu kết quả
        if not is_invertible:
            # Nếu Numpy báo không khả nghịch, hàm của bạn phải trả về string báo lỗi
            if isinstance(inv_f, str):
                print(f"Inverse: Đúng ở Test {i} (Đã bắt được lỗi: '{inv_f}')")
            else:
                print(f"Inverse: Sai ở Test {i} (Ma trận không vuông/suy biến nhưng hàm vẫn ra số)")
        else:
            # Nếu Numpy tính được, hàm của bạn phải ra mảng 2 chiều và khớp số liệu
            if isinstance(inv_f, str):
                print(f"Inverse: Sai ở Test {i} (Ma trận khả nghịch nhưng hàm lại báo lỗi: '{inv_f}')")
            else:
                try:
                    inv_f_np = np.array(inv_f, dtype=float)
                    # So sánh 2 ma trận với sai số epsilon
                    if np.allclose(inv_np, inv_f_np, atol=epsilon):
                        print(f"Inverse: Đúng ở Test {i}")
                    else:
                        print(f"Inverse: Sai ở Test {i} (Kết quả không khớp với Numpy)")
                except Exception as e:
                    print(f"Inverse: Sai ở Test {i} (Lỗi format list đầu ra: {e})")

In [62]:
def verify_rank_n_basic(rank_basicTestCase: list):
    epsilon = 1e-10
    epsilon_np = 1e-7

    def mat_vec_mul(A, v):
        return [sum(A[i][j] * v[j] for j in range(len(A[0]))) for i in range(len(A))]

    def is_null_vector(A, v, eps):
        res = mat_vec_mul(A, v)
        return all(abs(x) < eps for x in res)

    for i in range(len(rank_basicTestCase)):
        A = rank_basicTestCase[i]
        rankNP = rank_np(A)
        nullBasisNP = null_space_np(A)
        rankF, rowBasis, colBasis, nullBasis = rank_and_basis(A)

        if rankNP != rankF:
            print(f"Rank: Sai ở Test {i} (rank không khớp)")

        if len(nullBasisNP) > 0 and len(nullBasisNP[0]) != len(A[0]):
            nullBasisNP_vectors = list(zip(*nullBasisNP))
        else:
            nullBasisNP_vectors = nullBasisNP

        for idx, v in enumerate(nullBasisNP_vectors):
            v = list(v)
            if not is_null_vector(A, v, epsilon_np):
                print(f"Sai nullBasisNP ở Test {i}, vector {idx}")

        for idx, v in enumerate(nullBasis):
            if not is_null_vector(A, v, epsilon):
                print(f"Sai nullBasis ở Test {i}, vector {idx}")

        print(nullBasisNP)
        print(nullBasis)
        print(f"Đúng ở Test {i}")

In [63]:
def verify(LSTestCase: list, DetTestCase: list, rank_basicTestCase: list, InverseTestCase: list):
    """Bộ tham số đầu vào: LSTestcase: Để test giải HPT. Truyền vào dưới dạng List [A, b],
    Các list còn lại đều là 1 list[list], dùng để test đầy đủ các trường hợp"""
    epsilon = 1e-10

    print("\n--- TEST LEAST SQUARES ---")
    verify_LS(LSTestCase)
# ==========================================
    print("\n--- TEST INVERSE ---")
    verify_inverse(InverseTestCase)
# ==============================================
    print("\n--- TEST RANK BASIC ---")
    verify_rank_n_basic(rank_basicTestCase)


<b> BỘ TESTCASE MẪU ĐỦ CÁC TRƯỜNG HỢP </b> \
Bộ test được sinh bởi AI, đảm bảo đầy đủ các trường hợp


In [64]:

# 1. Testcase cho Hệ phương trình / Least Squares: list[(A, b)]
LSTestCase = [
    # Test 1: Hệ vuông, có nghiệm duy nhất (x1=1, x2=2)
    ( [[1, 2], [3, 4]], [5, 11] ),

    # Test 2: Hệ có vô số nghiệm (phương trình 2 là bội số của phương trình 1)
    ( [[1, 2, 3], [2, 4, 6]], [4, 8] ),

    # Test 3: Hệ vô nghiệm (mâu thuẫn: x+2y=3 và x+2y=5)
    ( [[1, 2], [1, 2]], [3, 5] ),

    # Test 4: Hệ Overdetermined (Nhiều phương trình hơn ẩn - Test bình phương tối thiểu)
    ( [[1, 1], [1, -1], [2, 1]], [2, 0, 3] ),

    #Test5
    ( [[8, 2], [3, -1], [5, 1]], [5, 1, 2] )
]

# 2. Testcase cho Định thức (Determinant): list[A]
DetTestCase = [
    [[5]],                                   # Test 1: Ma trận 1x1 (Det = 5)
    [[1, 2], [3, 4]],                        # Test 2: Ma trận 2x2 thông thường (Det = -2)
    [[1, 2, 3], [4, 5, 6], [7, 8, 9]],       # Test 3: Ma trận suy biến / phụ thuộc tuyến tính (Det = 0)
    [[2, 0, 0], [0, 3, 0], [0, 0, 4]],       # Test 4: Ma trận đường chéo (Det = 2*3*4 = 24)
    [[0, 1, 2], [1, 0, 3], [4, -3, 8]]       # Test 5: Ma trận có số 0 ở vị trí pivot đầu tiên (buộc phải hoán vị hàng)
]

# 3. Testcase cho Hạng ma trận (Rank)
rank_basicTestCase = [
    [[1, 2, 3], [4, 5, 6], [7, 8, 9]],       # Test 1: Hạng 2 (hàng 3 = 2*hàng 2 - hàng 1)
    [[1, 0, 0], [0, 1, 0], [0, 0, 0]],       # Test 2: Hạng 2 (ma trận đã ở dạng bậc thang)
    [[0, 0, 0], [0, 0, 0]],                  # Test 3: Hạng 0 (Ma trận toàn số 0)
    [[1, 2], [3, 4], [5, 6]],                # Test 4: Ma trận chữ nhật dọc (3x2) - Full rank = 2
    [[1, 2, 3, 4], [2, 4, 6, 8]]             # Test 5: Ma trận chữ nhật ngang (2x4) có các hàng tỷ lệ - Hạng 1
]

# 4. Testcase cho Ma trận nghịch đảo (Inverse): list[A]
InverseTestCase = [
    [[1, 2], [3, 4]],                        # Test 1: Khả nghịch 2x2 cơ bản
    [[1, 2, 3], [0, 1, 4], [5, 6, 0]],       # Test 2: Khả nghịch 3x3 thông thường
    [[1, 2, 3], [4, 5, 6], [7, 8, 9]],       # Test 3: Ma trận suy biến (Det = 0 -> Không có nghịch đảo)
    [[1, 0, 0], [0, 1, 0], [0, 0, 1]],       # Test 4: Ma trận đơn vị (Nghịch đảo là chính nó)
    [[2, 5], [1, 3]]                         # Test 5: Ma trận có định thức = 1 (Nghịch đảo sẽ ra số nguyên)
]

# GỌI HÀM KIỂM CHỨNG
verify(LSTestCase, DetTestCase, rank_basicTestCase, InverseTestCase)


--- TEST LEAST SQUARES ---
LS: Đúng ở Test 0
LS: Đúng ở Test 1
LS: Đúng ở Test 2
LS: Đúng ở Test 3
LS: Đúng ở Test 4

--- TEST INVERSE ---
Inverse: Đúng ở Test 0
Inverse: Đúng ở Test 1
Inverse: Đúng ở Test 2 (Đã bắt được lỗi: 'Ma tran suy bien, khong co nghich dao!')
Inverse: Đúng ở Test 3
Inverse: Đúng ở Test 4

--- TEST RANK BASIC ---
[[-0.49999999999999994, 1.0, -0.49999999999999994]]
[[1.0, -2.0, 1.0]]
Đúng ở Test 0
[[0.0, 0.0, 1.0]]
[[0.0, 0.0, 1.0]]
Đúng ở Test 1
[[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]]
[[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]]
Đúng ở Test 2
[]
[]
Đúng ở Test 3
[[1.0, -0.03355242085095901, -0.221491930496803, -0.06710484170191813], [0.1683392052895685, -0.8700530008264951, 1.0, -0.3570583009091446], [0.08878451517269072, 1.0, 0.527414365655154, -0.9177569030345382]]
[[-2.0, 1.0, 0.0, 0.0], [-3.0, 0.0, 1.0, 0.0], [-4.0, 0.0, 0.0, 1.0]]
Đúng ở Test 4
